# 04 — Preprocessing summary and cross-stage integrity review

**Purpose:** summarize and verify the full, auditable pathway `01 → 01b → 02 → 03`.

**Inputs:** the outputs of Notebook 01 (`dataset_clean.csv`, `data_errors.csv`, `error_examples.csv`), Notebook 01b (`dataset_review_clear.csv`, `dataset_review_suspected.csv`), Notebook 02 (`dataset_categorized.csv`) and Notebook 03 (`dataset_final.csv`, `dataset_excluded.csv`).

**Output:** `outputs/preprocessing_summary.md`.

**Protocol:** all records flagged by Notebook 01b are held in a separate **quality-control quarantine**, not declared erroneous. Only accepted records undergo categorization and eligibility evaluation. Quarantined records can be re-evaluated in a future audit; no source measurements are altered here. Notebook 03 exclusions represent study eligibility/remaining QC only.

This summary notebook does not change or re-export any CSV dataset.

In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display

OUTPUT_DIR = Path("../../outputs")
files = {
    "clean": "dataset_clean.csv",
    "errors": "data_errors.csv",
    "examples": "error_examples.csv",
    "accepted": "dataset_review_clear.csv",
    "quarantined": "dataset_review_suspected.csv",
    "categorized": "dataset_categorized.csv",
    "final": "dataset_final.csv",
    "excluded": "dataset_excluded.csv",
}

missing_files = [name for name in files.values() if not (OUTPUT_DIR / name).exists()]
if missing_files:
    raise FileNotFoundError(f"Missing preprocessing outputs: {missing_files}")

data = {name: pd.read_csv(OUTPUT_DIR / filename) for name, filename in files.items()}
clean, errors, examples = data["clean"], data["errors"], data["examples"]
accepted, quarantined = data["accepted"], data["quarantined"]
categorized, final, excluded = data["categorized"], data["final"], data["excluded"]

print("All eight preprocessing outputs loaded successfully.")

All eight preprocessing outputs loaded successfully.


## 1. Dataset flow and integrity checks across 01 → 01b → 02 → 03

In [2]:
# Verify full source partition and preserve data provenance across all stages.
for name, frame in [("clean", clean), ("accepted", accepted),
                    ("quarantined", quarantined), ("categorized", categorized),
                    ("final", final), ("excluded", excluded)]:
    if "source_excel_row" not in frame or frame["source_excel_row"].isna().any():
        raise ValueError(f"Missing source_excel_row in {name}")
    if not frame["source_excel_row"].is_unique:
        raise ValueError(f"Duplicated source_excel_row in {name}")

clean_ids = set(clean["source_excel_row"])
accepted_ids = set(accepted["source_excel_row"])
quarantined_ids = set(quarantined["source_excel_row"])
categorized_ids = set(categorized["source_excel_row"])
final_ids = set(final["source_excel_row"])
excluded_ids = set(excluded["source_excel_row"])

assert accepted_ids.isdisjoint(quarantined_ids), "Accepted and quarantined rows overlap"
assert accepted_ids | quarantined_ids == clean_ids, "Notebook 01b lost or added records"
assert categorized_ids == accepted_ids, "Notebook 02 must use only accepted 01b records"
assert final_ids.isdisjoint(excluded_ids), "A record is both retained and excluded"
assert final_ids | excluded_ids == categorized_ids, "Notebook 03 lost or added records"
assert final_ids.isdisjoint(quarantined_ids), "A quarantined record entered the final dataset"
assert final.columns.tolist() == categorized.columns.tolist(), "Final columns differ from categorized input"
assert set(clean.columns) == set(accepted.columns), "Accepted source fields differ from cleaned source"
assert set(clean.columns).issubset(quarantined.columns), "Quarantine data dropped source measurements"

# The 01b partition must preserve every cleaned measurement, not only row IDs.
from pandas.testing import assert_frame_equal
base_cols = clean.columns.tolist()
base = clean.set_index("source_excel_row").sort_index()
partition = pd.concat([accepted[base_cols], quarantined[base_cols]], ignore_index=True)
partition = partition.set_index("source_excel_row").sort_index()
assert_frame_equal(base, partition, check_dtype=False, check_exact=False,
                   rtol=0, atol=1e-10)

# Notebook 02 must not change accepted numerical fields.
accepted_base = accepted.set_index("source_excel_row").sort_index()
categorized_base = categorized[base_cols].set_index("source_excel_row").sort_index()
assert_frame_equal(accepted_base, categorized_base, check_dtype=False, check_exact=False,
                   rtol=0, atol=1e-10)

# Notebook 03 must not change categorized fields.
combined = pd.concat([final, excluded[categorized.columns]], ignore_index=True)
combined = combined.set_index("source_excel_row").sort_index()
categorized_compare = categorized.set_index("source_excel_row").sort_index()
assert_frame_equal(categorized_compare, combined, check_dtype=False, check_exact=False,
                   rtol=0, atol=1e-10)

flow = pd.DataFrame({
    "stage": ["01 — Cleaned source", "01b — Quarantined for review", "01b — Accepted for analysis",
              "02 — Categorized", "03 — Retained", "03 — Excluded by eligibility"],
    "records": [len(clean), len(quarantined), len(accepted), len(categorized), len(final), len(excluded)],
    "columns": [clean.shape[1], quarantined.shape[1], accepted.shape[1],
                categorized.shape[1], final.shape[1], excluded.shape[1]],
})
flow["percent_of_original"] = (100 * flow["records"] / len(clean)).round(1)

print("Data flow (quarantine and eligibility exclusions are disjoint):")
display(flow)
print(f"Final retention: {len(final)}/{len(clean)} ({len(final)/len(clean):.1%})")
print(f"Quarantined: {len(quarantined)}/{len(clean)} ({len(quarantined)/len(clean):.1%})")
print(f"Eligibility excluded: {len(excluded)}/{len(clean)} ({len(excluded)/len(clean):.1%})")

Data flow (quarantine and eligibility exclusions are disjoint):


,stage,records,columns,percent_of_original
0,01 — Cleaned source,530,19,100.0
1,01b — Quarantined for review,123,29,23.2
2,01b — Accepted for analysis,407,19,76.8
3,02 — Categorized,407,25,76.8
4,03 — Retained,402,25,75.8
5,03 — Excluded by eligibility,5,27,0.9


Final retention: 402/530 (75.8%)
Quarantined: 123/530 (23.2%)
Eligibility excluded: 5/530 (0.9%)


## 2. Notebook 01b audit and quarantine summary

In [3]:
required_review = ["review_reason_codes", "review_reason_count", "review_group_id"]
if any(column not in quarantined for column in required_review):
    raise ValueError(f"Quarantine file lacks required review metadata: {required_review}")

# Notebook 01b encodes multiple review reasons with a semicolon-space separator.
review_reasons = quarantined["review_reason_codes"].fillna("").str.split("; ", regex=False)
if not (review_reasons.str.len() == quarantined["review_reason_count"]).all():
    raise ValueError("Quarantine review_reason_count differs from review_reason_codes")
review_summary = (review_reasons.explode().value_counts()
                  .rename_axis("review_reason").reset_index(name="records"))

# 01b assigns group identifiers to both linked records and singleton review cases.
if "group_size" not in quarantined:
    raise ValueError("Missing group_size in quarantine audit metadata")
quarantine_groups = quarantined.groupby("review_group_id", dropna=False)
group_sizes = quarantine_groups["source_excel_row"].size()
if not (quarantine_groups["group_size"].first() == group_sizes).all():
    raise ValueError("Inconsistent group_size in quarantine metadata")
pair_groups = int((group_sizes > 1).sum())
individual_flags = int((group_sizes == 1).sum())
print(f"Pair/cluster groups: {pair_groups}; records not assigned to a pair/cluster: {individual_flags}")
print("Reasons for quarantine (not mutually exclusive):")
display(review_summary)

Pair/cluster groups: 45; records not assigned to a pair/cluster: 33
Reasons for quarantine (not mutually exclusive):


,review_reason,records
0,HF_VLF_SWAPPED,66
1,EXACT_SEVEN_HRV,16
2,MISSING_NUMERIC_MEASUREMENT,16
3,OUTSIDE_REVIEW_RANGE,14
4,HR_RR_FACTOR_TEN_PATTERN,11
5,SOURCE_BMI_DISAGREEMENT_CORRECTED,9
6,SOURCE_HEIGHT_DISAGREEMENT_CORRECTED,9
7,NEAR_HRV_MATCH,6
8,HR_RR_REVIEW_DISAGREEMENT,4
9,PARTIAL_HRV_MATCH,2


## 3. Cleaning and validation summary

In [4]:
# An issue is one log entry, not necessarily one affected participant.
issue_summary = (errors.groupby(["issue", "action"], dropna=False)
                 .size().reset_index(name="entries")
                 .sort_values("entries", ascending=False).reset_index(drop=True))

issue_by_column = (errors.groupby("column", dropna=False).size()
                   .reset_index(name="entries")
                   .sort_values("entries", ascending=False).reset_index(drop=True))

logged_rows = set(errors["source_excel_row"].dropna().astype(int))
example_rows = set(examples["source_excel_row"].astype(int))
if logged_rows != example_rows:
    raise ValueError("error_examples.csv does not match the rows recorded in data_errors.csv")

print(f"Log entries: {len(errors)}")
print(f"Participants with at least one logged issue: {len(logged_rows)}")
print(f"Rows in error_examples.csv: {len(examples)}")
print("\nEntries by issue type and action:")
display(issue_summary)
print("\nEntries by column:")
display(issue_by_column)
print("\nNote: issue counts can overlap within the same participant.")

Log entries: 183
Participants with at least one logged issue: 155
Rows in error_examples.csv: 155

Entries by issue type and action:


,issue,action,entries
0,text_number,converted_to_numeric,97
1,missing_value,kept_missing,20
2,identical_hrv_profile,review_no_change,16
3,outside_review_range,review_no_change,14
4,hr_rr_factor_10_pattern,review_no_change,11
5,bmi_disagreement,recomputed_from_weight_and_height,9
6,height_disagreement,corrected_from_height_cm,9
7,hr_rr_disagreement,review_no_change,4
8,empty_column,removed_column,2
9,empty_metadata_column,retained_for_traceability,1



Entries by column:


,column,entries
0,t2m_pre_rmssd,105
1,hrv_profile,16
2,t2m_pre_mean_hr/t2m_pre_mean_rr,15
3,t2m_pre_sdnn,14
4,t2m_pre_mean_rr,10
5,height_m,9
6,imc,9
7,t2m_pre_mean_hr,2
8,Unnamed: 0,1
9,Unnamed: 1,1



Note: issue counts can overlap within the same participant.


## 4. Factor categorization before and after filtering

In [5]:
factors = ["age_group", "sex_category", "sbp_category", "dbp_category", "bmi_category", "bmi_reference"]
missing_factors = [factor for factor in factors if factor not in categorized or factor not in final]
if missing_factors:
    raise ValueError(f"Missing category columns: {missing_factors}")

factor_tables = {}
for factor in factors:
    before = categorized[factor].fillna("Missing").value_counts(dropna=False)
    after = final[factor].fillna("Missing").value_counts(dropna=False)
    table = pd.DataFrame({"categorized_n": before, "retained_n": after}).fillna(0).astype(int)
    table["categorized_pct"] = (100 * table["categorized_n"] / len(categorized)).round(1)
    table["retained_pct"] = (100 * table["retained_n"] / len(final)).round(1)
    table.index.name = factor
    factor_tables[factor] = table.reset_index()
    print(f"\n{factor}")
    display(factor_tables[factor])


age_group


,age_group,categorized_n,retained_n,categorized_pct,retained_pct
0,60-69,184,184,45.2,45.8
1,70-79,190,190,46.7,47.3
2,80+,28,28,6.9,7.0
3,<60,5,0,1.2,0.0



sex_category


,sex_category,categorized_n,retained_n,categorized_pct,retained_pct
0,Female,299,294,73.5,73.1
1,Male,108,108,26.5,26.9



sbp_category


,sbp_category,categorized_n,retained_n,categorized_pct,retained_pct
0,<140 mmHg,303,298,74.4,74.1
1,>=140 mmHg,104,104,25.6,25.9


... (2 outputs left out)


bmi_category


,bmi_category,categorized_n,retained_n,categorized_pct,retained_pct
0,Obesity,154,150,37.8,37.3
1,Normal,118,118,29.0,29.4
2,Overweight,118,117,29.0,29.1
3,Underweight,17,17,4.2,4.2



bmi_reference


,bmi_reference,categorized_n,retained_n,categorized_pct,retained_pct
0,MINSAL adult (<60),5,0,1.2,0.0
1,MINSAL older person (>=60),402,402,98.8,100.0


## 5. Exclusion criteria and multiple reasons

In [6]:
needed = ["exclusion_count", "exclusion_reasons"]
if any(name not in excluded for name in needed):
    raise ValueError("Exclusion count/reasons columns are missing")

# Split the exact separator written by Notebook 03.
all_reasons = excluded["exclusion_reasons"].fillna("").str.split(" | ", regex=False)
if not (all_reasons.str.len() == excluded["exclusion_count"]).all():
    raise ValueError("Some exclusion_count values do not match the listed reasons")

reason_summary = (all_reasons.explode().value_counts()
                  .rename_axis("reason").reset_index(name="participants"))
reason_summary["percent_of_excluded"] = (
    100 * reason_summary["participants"] / len(excluded)
).round(1)

multiplicity = (excluded["exclusion_count"].value_counts().sort_index()
                .rename_axis("reasons_per_participant")
                .reset_index(name="participants"))

reason_combinations = (excluded["exclusion_reasons"].value_counts()
                       .rename_axis("combined_reasons").reset_index(name="participants"))

print("Exclusion reasons (not mutually exclusive):")
display(reason_summary)
print("\nNumber of distinct reasons per excluded participant:")
display(multiplicity)
print("\nObserved combinations of exclusion reasons:")
display(reason_combinations)
print(f"\nExcluded participants with multiple reasons: {(excluded['exclusion_count'] > 1).sum()}")

Exclusion reasons (not mutually exclusive):


,reason,participants,percent_of_excluded
0,AGE_BELOW_60,5,100.0



Number of distinct reasons per excluded participant:


,reasons_per_participant,participants
0,1,5



Observed combinations of exclusion reasons:


,combined_reasons,participants
0,AGE_BELOW_60,5



Excluded participants with multiple reasons: 0


## 6. Final sample composition and factorial group sizes

In [7]:
age_by_sex = pd.crosstab(final["age_group"], final["sex_category"])
sbp_by_bmi = pd.crosstab(final["sbp_category"], final["bmi_category"])
dbp_by_bmi = pd.crosstab(final["dbp_category"], final["bmi_category"])

# Each pressure measurement defines a separate four-factor analysis.
# The two pressure categories must not be merged into a new factor.
four_factor_counts = {}
for pressure_factor in ["sbp_category", "dbp_category"]:
    four_factor_counts[pressure_factor] = (
        final.groupby(
            ["age_group", "sex_category", pressure_factor, "bmi_category"],
            dropna=False
        ).size().reset_index(name="participants")
    )

print("Age group × sex (retained sample):")
display(age_by_sex)
print("\nSystolic BP category × BMI category (retained sample):")
display(sbp_by_bmi)
print("\nDiastolic BP category × BMI category (retained sample):")
display(dbp_by_bmi)

for pressure_factor, counts in four_factor_counts.items():
    print(f"\nAge × sex × {pressure_factor} × BMI")
    print("Observed combinations:", len(counts))
    print("Combinations with fewer than 5 participants:", (counts["participants"] < 5).sum())
    print("Combinations with fewer than 10 participants:", (counts["participants"] < 10).sum())
    print("Smallest observed four-factor groups:")
    display(counts.sort_values("participants").head(15))

hrv_columns = ["t2m_pre_mean_rr", "t2m_pre_mean_hr", "t2m_pre_sdnn",
               "t2m_pre_rmssd", "t2m_pre_hf", "t2m_pre_lf", "t2m_pre_vlf"]
missing_hrv = final[hrv_columns].isna().sum().rename("missing_n").reset_index()
missing_hrv.columns = ["hrv_variable", "missing_n"]
print("\nRemaining missing HRV values:")
display(missing_hrv)

Age group × sex (retained sample):


sex_category,Female,Male
age_group,,
60-69,141,43
70-79,131,59
80+,22,6



Systolic BP category × BMI category (retained sample):


bmi_category,Normal,Obesity,Overweight,Underweight
sbp_category,,,,
<140 mmHg,90,105,89,14
>=140 mmHg,28,45,28,3



Diastolic BP category × BMI category (retained sample):


bmi_category,Normal,Obesity,Overweight,Underweight
dbp_category,,,,
<90 mmHg,109,126,102,17
>=90 mmHg,9,24,15,0


... (2 outputs left out)


Age × sex × dbp_category × BMI
Observed combinations: 33
Combinations with fewer than 5 participants: 13
Combinations with fewer than 10 participants: 21
Smallest observed four-factor groups:


,age_group,sex_category,dbp_category,bmi_category,participants
11,60-69,Male,>=90 mmHg,Normal,1
29,80+,Female,<90 mmHg,Underweight,1
31,80+,Male,<90 mmHg,Obesity,1
23,70-79,Male,>=90 mmHg,Normal,1
25,70-79,Male,>=90 mmHg,Overweight,2
10,60-69,Male,<90 mmHg,Underweight,2
30,80+,Male,<90 mmHg,Normal,2
13,60-69,Male,>=90 mmHg,Overweight,3
24,70-79,Male,>=90 mmHg,Obesity,3
27,80+,Female,<90 mmHg,Obesity,3



Remaining missing HRV values:


,hrv_variable,missing_n
0,t2m_pre_mean_rr,0
1,t2m_pre_mean_hr,0
2,t2m_pre_sdnn,0
3,t2m_pre_rmssd,0
4,t2m_pre_hf,0
5,t2m_pre_lf,0
6,t2m_pre_vlf,0


## 7. Export consolidated summary

In [8]:
# A single readable report; no dataset is rewritten.
report = [
    "# HRV preprocessing summary",
    "",
    "Based on the outputs of Notebooks 01, 01b, 02, and 03.",
    "",
    "## Dataset flow",
    flow.to_markdown(index=False),
    "",
    f"Final retention: {len(final)}/{len(clean)} ({len(final)/len(clean):.1%}) of original records.",
    f"Quarantined for quality review: {len(quarantined)}/{len(clean)} ({len(quarantined)/len(clean):.1%}).",
    f"Eligibility exclusions after 01b: {len(excluded)}/{len(clean)} ({len(excluded)/len(clean):.1%}).",
    "Quarantine does not confirm errors. Suspicious records remain available for future review.",
    "",
    "## Notebook 01b quality-review quarantine",
    f"Groups with paired/clustered records: {pair_groups}; singleton review records: {individual_flags}.",
    "Review-reason frequencies overlap and must not be summed.",
    review_summary.to_markdown(index=False),
    "",
    "## Cleaning and validation",
    f"Log entries: {len(errors)}. Unique participants with logged issues: {len(logged_rows)}.",
    f"Original error examples: {len(examples)} rows.",
    "",
    issue_summary.to_markdown(index=False),
    "",
    "### Issues by column",
    issue_by_column.to_markdown(index=False),
    "",
    "## Factor categories",
]
for factor, table in factor_tables.items():
    report.extend(["", f"### {factor}", table.to_markdown(index=False)])

report.extend([
    "", "## Eligibility exclusions after quality quarantine",
    "Reasons are not mutually exclusive; counts should not be summed to obtain excluded participants.",
    reason_summary.to_markdown(index=False),
    "", "### Number of reasons per participant",
    multiplicity.to_markdown(index=False),
    "", "### Observed exclusion combinations",
    reason_combinations.assign(
        combined_reasons=lambda frame: frame["combined_reasons"].str.replace("|", r"\|", regex=False)
    ).to_markdown(index=False),
    "", "## Final sample: age by sex",
    age_by_sex.to_markdown(),
    "", "## Final sample: systolic BP by BMI",
    sbp_by_bmi.to_markdown(),
    "", "## Final sample: diastolic BP by BMI",
    dbp_by_bmi.to_markdown(),
    "", "## Four-factor combinations",
    "Age × sex × pressure × BMI is summarized separately for systolic and diastolic pressure.",
])

for pressure_factor, counts in four_factor_counts.items():
    report.extend([
        "",
        f"### Age × sex × {pressure_factor} × BMI",
        f"Observed combinations: {len(counts)}; "
        f"fewer than five participants: {(counts['participants'] < 5).sum()}; "
        f"fewer than ten: {(counts['participants'] < 10).sum()}.",
        "",
        "Smallest observed groups:",
        counts.sort_values("participants").head(15).to_markdown(index=False),
    ])

report.extend([
    "", "## Missing HRV values in the final sample",
    missing_hrv.to_markdown(index=False),
    "",
])

SUMMARY_FILE = OUTPUT_DIR / "preprocessing_summary.md"
SUMMARY_FILE.write_text("\n".join(report), encoding="utf-8")
print(f"Saved: {SUMMARY_FILE}")
print("All input datasets remain unchanged.")

Saved: ../../outputs/preprocessing_summary.md
All input datasets remain unchanged.
